<a href="https://colab.research.google.com/github/kavks03-bit/E-COMMERCE-ANALYSIS-INTELLIGENT-SYSTEM/blob/main/MINI_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#===============================================
#E_COMMERCE ANALYSIS INTELLIGENT SYSTEM
#===============================================
#Project Flow:
# Data Generation
#       ↓
# Data Cleaning
#       ↓
# EDA
#       ↓
# Sales Analytics
#       ↓
# Customer Analytics
#       ↓
# RFM Analysis
#       ↓
# Customer Segmentation
#       ↓
# K-Means Clustering
#       ↓
# Interactive Analysis
#       ↓
# Business Insights
# ============================================================

#1. INSTALL / IMPORT LIBRARIES
# ============================================================

#!pip install -q pandas numpy plotly scikit-learn ipywidgets

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import ipywidgets as widgets
from IPython.display import display, clear_output

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 2. CREATE E-COMMERCE DATASET
# ============================================================

np.random.seed(42)

# Number of transactions
n_transactions = 10000

# Customer IDs
customer_ids = np.random.randint(
    1001,
    2001,
    n_transactions
)

# Products
products = [
    "Laptop",
    "Smartphone",
    "Headphones",
    "Smartwatch",
    "Tablet",
    "Keyboard",
    "Mouse",
    "Monitor",
    "Camera",
    "Speaker"
]

# Product categories
categories = {
    "Laptop": "Electronics",
    "Smartphone": "Electronics",
    "Headphones": "Accessories",
    "Smartwatch": "Wearables",
    "Tablet": "Electronics",
    "Keyboard": "Accessories",
    "Mouse": "Accessories",
    "Monitor": "Electronics",
    "Camera": "Electronics",
    "Speaker": "Audio"
}

# Countries
countries = [
    "India",
    "USA",
    "UK",
    "Germany",
    "Canada",
    "Australia"
]

# Generate product
product_data = np.random.choice(
    products,
    n_transactions
)

# Generate quantity
quantity_data = np.random.randint(
    1,
    6,
    n_transactions
)

# Generate prices
price_data = np.random.uniform(
    20,
    1500,
    n_transactions
)

# Generate dates
date_data = pd.date_range(
    start="2025-01-01",
    end="2025-12-31",
    periods=n_transactions
)

# Create DataFrame
df = pd.DataFrame({

    "Customer_ID": customer_ids,

    "Product": product_data,

    "Category": [
        categories[p]
        for p in product_data
    ],

    "Country": np.random.choice(
        countries,
        n_transactions
    ),

    "Quantity": quantity_data,

    "UnitPrice": price_data,

    "PurchaseDate": date_data
})


# ============================================================
# 3. CREATE REVENUE
# ============================================================

df["Revenue"] = (
    df["Quantity"] *
    df["UnitPrice"]
)


# ============================================================
# 4. ADD ADDITIONAL CUSTOMER INFORMATION
# ============================================================

customer_gender = {}

for customer in df["Customer_ID"].unique():

    customer_gender[customer] = np.random.choice(
        ["Male", "Female"],
        p=[0.55, 0.45]
    )

df["Gender"] = df["Customer_ID"].map(
    customer_gender
)


# ============================================================
# 5. DISPLAY DATASET
# ============================================================

print("=" * 60)
print("ECOMMERCE DATASET")
print("=" * 60)

print("Dataset Shape:")
print(df.shape)

print("First ten records:")
print(df.head(10))


# ============================================================
# 6. DATA INFORMATION
# ============================================================

print("=" * 60)
print("DATA INFORMATION")
print("=" * 60)

df.info()


# ============================================================
# 7. MISSING VALUES
# ============================================================

print("=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing_values = df.isnull().sum()

display(
    missing_values.to_frame(
        "Missing values"
    )
)


# ============================================================
# 8. CHECK DUPLICATES
# ============================================================

print("=" * 60)
print("CHECK DUPLICATES")
print("=" * 60)

duplicates = df.duplicated().sum()

print(
    "Number of duplicate records:",
    duplicates
)


# ============================================================
# 9. DATA CLEANING
# ============================================================

df = df.drop_duplicates()

df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
]

df["PurchaseDate"] = pd.to_datetime(
    df["PurchaseDate"]
)

df["Revenue"] = (
    df["Quantity"] *
    df["UnitPrice"]
)


# ============================================================
# 10. DATE FEATURES
# ============================================================

df["Year"] = (
    df["PurchaseDate"].dt.year
)

df["Month"] = (
    df["PurchaseDate"].dt.month
)

df["MonthName"] = (
    df["PurchaseDate"]
    .dt.strftime("%B")
)

df["Day"] = (
    df["PurchaseDate"].dt.day
)

df["Weekday"] = (
    df["PurchaseDate"].dt.day_name()
)


# ============================================================
# 11. BUSINESS KPIs
# ============================================================

total_revenue = df["Revenue"].sum()

total_customers = df[
    "Customer_ID"
].nunique()

total_transactions = len(df)

total_units = df[
    "Quantity"
].sum()

average_transaction_value = (
    df["Revenue"].mean()
)

print("=" * 60)
print("BUSINESS KPIs")
print("=" * 60)

print(
    f"Total Revenue              : ${total_revenue:,.2f}"
)

print(
    f"Total Customers            : {total_customers:,}"
)

print(
    f"Total Transactions         : {total_transactions:,}"
)

print(
    f"Total Units Sold           : {total_units:,}"
)

print(
    f"Average Transaction Value  : ${average_transaction_value:,.2f}"
)


# ============================================================
# 12. MONTHLY REVENUE
# ============================================================

monthly_revenue = (
    df.groupby("MonthName")["Revenue"]
    .sum()
    .reset_index()
)

month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

monthly_revenue["MonthName"] = pd.Categorical(
    monthly_revenue["MonthName"],
    categories=month_order,
    ordered=True
)

monthly_revenue = monthly_revenue.sort_values(
    "MonthName"
)

fig = px.line(
    monthly_revenue,
    x="MonthName",
    y="Revenue",
    markers=True,
    title="Monthly Revenue Trend"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Revenue"
)

fig.show()


# ============================================================
# 13. PRODUCT REVENUE
# ============================================================

product_revenue = (
    df.groupby("Product")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.bar(
    product_revenue,
    x="Revenue",
    y="Product",
    orientation="h",
    title="Revenue by Product"
)

fig.show()


# ============================================================
# 14. PRODUCT QUANTITY ANALYSIS
# ============================================================

product_quantity = (
    df.groupby("Product")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.bar(
    product_quantity,
    x="Product",
    y="Quantity",
    title="Units Sold by Product"
)

fig.show()


# ============================================================
# 15. CATEGORY ANALYSIS
# ============================================================

category_revenue = (
    df.groupby("Category")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.pie(
    category_revenue,
    names="Category",
    values="Revenue",
    hole=0.4,
    title="Revenue by Category"
)

fig.show()


# ============================================================
# 16. COUNTRY ANALYSIS
# ============================================================

country_revenue = (
    df.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.bar(
    country_revenue,
    x="Country",
    y="Revenue",
    title="Revenue by Country"
)

fig.show()


# ============================================================
# 17. CUSTOMER REVENUE
# ============================================================

customer_revenue = (
    df.groupby("Customer_ID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

print("=" * 60)
print("TOP 10 CUSTOMERS")
print("=" * 60)

display(
    customer_revenue.head(10)
)


# ============================================================
# 18. TOP 10 CUSTOMERS VISUALIZATION
# ============================================================

top_customers = customer_revenue.head(10)

fig = px.bar(
    top_customers,
    x="Customer_ID",
    y="Revenue",
    title="Top 10 Customers by Revenue"
)

fig.show()


# ============================================================
# 19. CUSTOMER PURCHASE FREQUENCY
# ============================================================

customer_frequency = (
    df.groupby("Customer_ID")
    .size()
    .reset_index(
        name="Frequency"
    )
)

fig = px.histogram(
    customer_frequency,
    x="Frequency",
    nbins=30,
    title="Customer Purchase Frequency"
)

fig.show()


# ============================================================
# 20. RFM ANALYSIS
# ============================================================

print("=" * 60)
print("RFM ANALYSIS")
print("=" * 60)


# Reference date
reference_date = (
    df["PurchaseDate"].max()
    + pd.Timedelta(days=1)
)


# Calculate RFM
rfm = df.groupby(
    "Customer_ID"
).agg({

    "PurchaseDate":
        lambda x:
        (
            reference_date -
            x.max()
        ).days,

    "Customer_ID":
        "count",

    "Revenue":
        "sum"
})


# Rename columns
rfm.columns = [
    "Recency",
    "Frequency",
    "Monetary"
]

rfm = rfm.reset_index()


print("\n========== RFM VALUES ==========")

display(
    rfm.head(10)
)


# ============================================================
# RFM SCORING
# ============================================================

# Recency
# Lower Recency is better
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)


# Frequency
# Higher Frequency is better
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


# Monetary
# Higher Monetary value is better
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


# Combine R, F and M scores
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)


print("\n========== RFM SCORES ==========")

display(
    rfm[
        [
            "Customer_ID",
            "Recency",
            "Frequency",
            "Monetary",
            "R_Score",
            "F_Score",
            "M_Score",
            "RFM_Score"
        ]
    ].head(10)
)


# ============================================================
# 21. CUSTOMER SEGMENTS USING RFM
# ============================================================

def assign_segment(row):

    if (
        row["R_Score"] >= 4
        and row["F_Score"] >= 4
        and row["M_Score"] >= 4
    ):

        return "Champions"

    elif (
        row["R_Score"] >= 4
        and row["F_Score"] >= 3
    ):

        return "Loyal Customers"

    elif (
        row["R_Score"] >= 4
        and row["M_Score"] >= 3
    ):

        return "Potential Loyalists"

    elif (
        row["R_Score"] <= 2
        and row["F_Score"] >= 3
    ):

        return "At Risk"

    elif (
        row["R_Score"] <= 2
        and row["M_Score"] >= 3
    ):

        return "High Value At Risk"

    elif (
        row["R_Score"] <= 2
        and row["F_Score"] <= 2
    ):

        return "Lost Customers"

    else:

        return "Regular Customers"


rfm["Segment"] = rfm.apply(
    assign_segment,
    axis=1
)


print("\n========== CUSTOMER SEGMENTS ==========")

segment_count = (
    rfm["Segment"]
    .value_counts()
    .reset_index()
)

segment_count.columns = [
    "Segment",
    "CustomerCount"
]

display(
    segment_count
)


# ============================================================
# 22. SEGMENT DISTRIBUTION
# ============================================================

fig = px.pie(
    segment_count,
    names="Segment",
    values="CustomerCount",
    hole=0.4,
    title="Customer Segment Distribution"
)

fig.show()


# ============================================================
# 23. 3D RFM VISUALIZATION
# ============================================================

fig = px.scatter_3d(
    rfm,
    x="Recency",
    y="Frequency",
    z="Monetary",
    color="Segment",

    hover_data=[
        "Customer_ID",
        "RFM_Score"
    ],

    title="3D RFM Customer Analysis"
)

fig.update_layout(
    scene=dict(
        xaxis_title="Recency",
        yaxis_title="Frequency",
        zaxis_title="Monetary"
    )
)

fig.show()

# ============================================================
# 24. REVENUE BY CUSTOMER SEGMENT
# ============================================================

segment_revenue = (
    rfm.groupby("Segment")["Monetary"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.bar(
    segment_revenue,
    x="Segment",
    y="Monetary",
    title="Revenue Contribution by Customer Segment"
)

fig.show()


# ============================================================
# 25. CUSTOMER BEHAVIOUR
# ============================================================

fig = px.scatter(
    rfm,
    x="Frequency",
    y="Monetary",
    size="Monetary",
    color="Segment",

    hover_data=[
        "Customer_ID",
        "Recency",
        "RFM_Score"
    ],

    title="Customer Frequency vs Spending"
)

fig.show()


# ============================================================
# 26. RFM 3D VISUALIZATION
# ============================================================

fig = px.scatter_3d(
    rfm,
    x="Recency",
    y="Frequency",
    z="Monetary",
    color="Segment",

    hover_data=[
        "Customer_ID"
    ],

    title="3D RFM Customer Analysis"
)

fig.update_layout(
    scene=dict(
        xaxis_title="Recency",
        yaxis_title="Frequency",
        zaxis_title="Monetary"
    )
)

fig.show()


# ============================================================
# 27. K-MEANS CUSTOMER SEGMENTATION
# ============================================================

print("=" * 60)
print("K-MEANS CUSTOMER SEGMENTATION")
print("=" * 60)


# ------------------------------------------------------------
# Select features for K-Means
# ------------------------------------------------------------

features = rfm[
    [
        "Recency",
        "Frequency",
        "Monetary"
    ]
]


# ------------------------------------------------------------
# Standardize the features
# ------------------------------------------------------------

scaler = StandardScaler()

scaled_features = scaler.fit_transform(
    features
)


# ------------------------------------------------------------
# Create K-Means model
# ------------------------------------------------------------

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)


# ------------------------------------------------------------
# Train K-Means
# ------------------------------------------------------------

kmeans.fit(
    scaled_features
)


# ------------------------------------------------------------
# Assign cluster to each customer
# ------------------------------------------------------------

rfm["Cluster"] = kmeans.labels_


print("\n========== K-MEANS RESULTS ==========")

display(
    rfm[
        [
            "Customer_ID",
            "Recency",
            "Frequency",
            "Monetary",
            "Cluster"
        ]
    ].head(10)
)


# ============================================================
# 27.1 K-MEANS CLUSTER VISUALIZATION
# ============================================================

fig = px.scatter_3d(
    rfm,
    x="Recency",
    y="Frequency",
    z="Monetary",
    color="Cluster",

    hover_data=[
        "Customer_ID",
        "Segment",
        "RFM_Score"
    ],

    title="K-Means Customer Segmentation - 3D"
)

fig.update_layout(
    scene=dict(
        xaxis_title="Recency",
        yaxis_title="Frequency",
        zaxis_title="Monetary"
    )
)

fig.show()

# ============================================================
# 28. CLUSTER SUMMARY
# ============================================================

print("=" * 60)
print("K-MEANS CLUSTER SUMMARY")
print("=" * 60)


cluster_summary = (
    rfm.groupby("Cluster")
    .agg({
        "Recency": "mean",
        "Frequency": "mean",
        "Monetary": "mean",
        "Customer_ID": "count"
    })
    .round(2)
)


cluster_summary.rename(
    columns={
        "Customer_ID": "Customers"
    },
    inplace=True
)


print("Cluster Summary:")

display(
    cluster_summary
)


# ============================================================
# 29. K-MEANS 3D VISUALIZATION
# ============================================================

fig = px.scatter_3d(
    rfm,
    x="Recency",
    y="Frequency",
    z="Monetary",
    color="Cluster",

    hover_data=[
        "Customer_ID",
        "Segment"
    ],

    title="K-Means Customer Clusters"
)

fig.update_layout(
    scene=dict(
        xaxis_title="Recency",
        yaxis_title="Frequency",
        zaxis_title="Monetary"
    )
)

fig.show()


# ============================================================
# 30. FIND AT-RISK CUSTOMERS
# ============================================================

at_risk = rfm[
    rfm["Segment"] == "At Risk"
]


print("=" * 60)
print("AT-RISK CUSTOMERS")
print("=" * 60)

print(
    "Number of At-Risk Customers:",
    len(at_risk)
)


display(
    at_risk[
        [
            "Customer_ID",
            "Recency",
            "Frequency",
            "Monetary",
            "RFM_Score"
        ]
    ]
    .sort_values(
        "Monetary",
        ascending=False
    )
    .head(20)
)


# ============================================================
# 31. CHAMPION CUSTOMERS
# ============================================================

champions = rfm[
    rfm["Segment"] == "Champions"
]


print("=" * 60)
print("CHAMPION CUSTOMERS")
print("=" * 60)

print(
    "Number of Champions:",
    len(champions)
)


display(
    champions[
        [
            "Customer_ID",
            "Recency",
            "Frequency",
            "Monetary",
            "RFM_Score"
        ]
    ]
    .sort_values(
        "Monetary",
        ascending=False
    )
    .head(20)
)


# ============================================================
# 32. INTERACTIVE PRODUCT ANALYSIS
# ============================================================

print("=" * 60)
print("INTERACTIVE PRODUCT ANALYSIS")
print("=" * 60)


product_dropdown = widgets.Dropdown(

    options=[
        "All Products"
    ] +
    sorted(
        df["Product"].unique()
    ),

    description="Product:",

    style={
        "description_width": "initial"
    }
)


product_output = widgets.Output()


def product_analysis(change):

    with product_output:

        clear_output(wait=True)

        selected_product = change["new"]


        # ----------------------------------------------------
        # Filter data
        # ----------------------------------------------------

        if selected_product == "All Products":

            temp = df.copy()

        else:

            temp = df[
                df["Product"] == selected_product
            ]


        # ----------------------------------------------------
        # KPIs
        # ----------------------------------------------------

        revenue = temp[
            "Revenue"
        ].sum()

        quantity = temp[
            "Quantity"
        ].sum()

        transactions = len(temp)


        print(
            f"Product: {selected_product}"
        )

        print(
            f"Revenue: ${revenue:,.2f}"
        )

        print(
            f"Units Sold: {quantity:,}"
        )

        print(
            f"Transactions: {transactions:,}"
        )


        # ----------------------------------------------------
        # Monthly analysis
        # ----------------------------------------------------

        monthly = (
            temp.groupby(
                "MonthName"
            )["Revenue"]
            .sum()
            .reset_index()
        )


        monthly["MonthName"] = pd.Categorical(
            monthly["MonthName"],
            categories=month_order,
            ordered=True
        )


        monthly = monthly.sort_values(
            "MonthName"
        )


        fig = px.line(
            monthly,
            x="MonthName",
            y="Revenue",
            markers=True,
            title=f"Monthly Revenue - {selected_product}"
        )


        fig.show()


product_dropdown.observe(
    product_analysis,
    names="value"
)


display(
    product_dropdown
)

display(
    product_output
)


# ============================================================
# 33. INTERACTIVE COUNTRY ANALYSIS
# ============================================================

print("=" * 60)
print("INTERACTIVE COUNTRY ANALYSIS")
print("=" * 60)


country_dropdown = widgets.Dropdown(

    options=[
        "All Countries"
    ] +
    sorted(
        df["Country"].unique()
    ),

    description="Country:",

    style={
        "description_width": "initial"
    }
)


country_output = widgets.Output()


def country_analysis(change):

    with country_output:

        clear_output(wait=True)

        selected_country = change["new"]


        # ----------------------------------------------------
        # Filter data
        # ----------------------------------------------------

        if selected_country == "All Countries":

            temp = df.copy()

        else:

            temp = df[
                df["Country"] == selected_country
            ]


        # ----------------------------------------------------
        # KPIs
        # ----------------------------------------------------

        revenue = temp[
            "Revenue"
        ].sum()

        customers = temp[
            "Customer_ID"
        ].nunique()

        transactions = len(temp)


        print(
            f"Country: {selected_country}"
        )

        print(
            f"Revenue: ${revenue:,.2f}"
        )

        print(
            f"Customers: {customers:,}"
        )

        print(
            f"Transactions: {transactions:,}"
        )


        # ----------------------------------------------------
        # Category analysis
        # ----------------------------------------------------

        category = (
            temp.groupby(
                "Category"
            )["Revenue"]
            .sum()
            .reset_index()
        )


        fig = px.bar(
            category,
            x="Category",
            y="Revenue",
            title=f"Category Performance - {selected_country}"
        )


        fig.show()


country_dropdown.observe(
    country_analysis,
    names="value"
)


display(
    country_dropdown
)

display(
    country_output
)


# ============================================================
# 34. INTERACTIVE CUSTOMER SEGMENT FILTER
# ============================================================

print("=" * 60)
print("CUSTOMER SEGMENT EXPLORER")
print("=" * 60)


segment_dropdown = widgets.Dropdown(

    options=[
        "All Segments"
    ] +
    sorted(
        rfm["Segment"].unique()
    ),

    description="Segment:",

    style={
        "description_width": "initial"
    }
)


segment_output = widgets.Output()


def segment_analysis(change):

    with segment_output:

        clear_output(wait=True)

        selected_segment = change["new"]


        # ----------------------------------------------------
        # Filter data
        # ----------------------------------------------------

        if selected_segment == "All Segments":

            temp = rfm.copy()

        else:

            temp = rfm[
                rfm["Segment"] == selected_segment
            ]


        # ----------------------------------------------------
        # Segment KPIs
        # ----------------------------------------------------

        print(
            f"Segment: {selected_segment}"
        )

        print(
            f"Customers: {len(temp):,}"
        )

        print(
            f"Average Recency: "
            f"{temp['Recency'].mean():.2f} days"
        )

        print(
            f"Average Frequency: "
            f"{temp['Frequency'].mean():.2f}"
        )

        print(
            f"Total Revenue: "
            f"${temp['Monetary'].sum():,.2f}"
        )


        # ----------------------------------------------------
        # Customer table
        # ----------------------------------------------------

        display(
            temp[
                [
                    "Customer_ID",
                    "Recency",
                    "Frequency",
                    "Monetary",
                    "RFM_Score",
                    "Segment"
                ]
            ]
            .sort_values(
                "Monetary",
                ascending=False
            )
            .head(20)
        )


segment_dropdown.observe(
    segment_analysis,
    names="value"
)


display(
    segment_dropdown
)

display(
    segment_output
)


# ============================================================
# 35. AUTOMATIC BUSINESS INSIGHTS
# ============================================================

print("=" * 60)
print("AUTOMATIC BUSINESS INSIGHTS")
print("=" * 60)


# ------------------------------------------------------------
# Best Product
# ------------------------------------------------------------

best_product = (
    df.groupby("Product")["Revenue"]
    .sum()
    .idxmax()
)


# ------------------------------------------------------------
# Best Country
# ------------------------------------------------------------

best_country = (
    df.groupby("Country")["Revenue"]
    .sum()
    .idxmax()
)


# ------------------------------------------------------------
# Best Category
# ------------------------------------------------------------

best_category = (
    df.groupby("Category")["Revenue"]
    .sum()
    .idxmax()
)


# ------------------------------------------------------------
# Largest Customer Segment
# ------------------------------------------------------------

largest_segment = (
    rfm["Segment"]
    .value_counts()
    .idxmax()
)


print(
    f"Best Product    : {best_product}"
)

print(
    f"Best Country    : {best_country}"
)

print(
    f"Best Category   : {best_category}"
)

print(
    f"Largest Segment : {largest_segment}"
)


# ============================================================
# 36. FINAL CUSTOMER INTELLIGENCE REPORT
# ============================================================

print("=" * 60)
print("FINAL E-COMMERCE CUSTOMER INTELLIGENCE REPORT")
print("=" * 60)

print()

print(
    f"Total Revenue       : ${total_revenue:,.2f}"
)

print(
    f"Total Customers     : {total_customers:,}"
)

print(
    f"Total Transactions  : {total_transactions:,}"
)

print(
    f"Total Units Sold    : {total_units:,}"
)

print(
    f"Average Order Value : ${average_transaction_value:,.2f}"
)

print()

print(
    f"Best Product        : {best_product}"
)

print(
    f"Best Country        : {best_country}"
)

print(
    f"Best Category       : {best_category}"
)

print(
    f"Largest Segment     : {largest_segment}"
)

print()

print(
    "Customer Intelligence Analysis Completed."
)

print("=" * 60)

ECOMMERCE DATASET
Dataset Shape:
(10000, 9)
First ten records:
   Customer_ID     Product     Category  Country  Quantity    UnitPrice  \
0         1103     Speaker        Audio  Germany         5   421.220516   
1         1436       Mouse  Accessories  Germany         5  1060.482352   
2         1861    Keyboard  Accessories      USA         3  1394.133606   
3         1271      Laptop  Electronics  Germany         5  1035.704409   
4         1107     Monitor  Electronics       UK         4  1028.042058   
5         1072  Smartphone  Electronics    India         2   351.183402   
6         1701      Tablet  Electronics  Germany         2   811.992696   
7         1021      Camera  Electronics       UK         4   852.533809   
8         1615     Monitor  Electronics       UK         5  1056.191752   
9         1122  Smartphone  Electronics      USA         1  1495.295746   

                   PurchaseDate      Revenue  Gender  
0 2025-01-01 00:00:00.000000000  2106.102579  Female  
1

,Missing values
Customer_ID,0
Product,0
Category,0
Country,0
Quantity,0
UnitPrice,0
PurchaseDate,0
Revenue,0
Gender,0


CHECK DUPLICATES
Number of duplicate records: 0
BUSINESS KPIs
Total Revenue              : $22,779,760.07
Total Customers            : 1,000
Total Transactions         : 10,000
Total Units Sold           : 30,054
Average Transaction Value  : $2,277.98


TOP 10 CUSTOMERS


,Customer_ID,Revenue
0,1073,63411.273057
1,1726,54753.396789
2,1936,52605.319195
3,1758,51151.289868
4,1889,50105.289656
5,1283,49445.197724
6,1929,49419.019015
7,1659,48800.421761
8,1130,47903.580832
9,1891,46723.822640


...

Output()

CUSTOMER SEGMENT EXPLORER


Dropdown(description='Segment:', options=('All Segments', 'At Risk', 'Champions', 'High Value At Risk', 'Lost …

Output()

AUTOMATIC BUSINESS INSIGHTS
Best Product    : Speaker
Best Country    : Germany
Best Category   : Electronics
Largest Segment : Regular Customers
FINAL E-COMMERCE CUSTOMER INTELLIGENCE REPORT

Total Revenue       : $22,779,760.07
Total Customers     : 1,000
Total Transactions  : 10,000
Total Units Sold    : 30,054
Average Order Value : $2,277.98

Best Product        : Speaker
Best Country        : Germany
Best Category       : Electronics
Largest Segment     : Regular Customers

Customer Intelligence Analysis Completed.
